# Même expérience que le notebook BPTT — mais dans **Genesis**

Reprise de `exact_residual_finetune_cf.ipynb`, mais le « quad lourd réel » n'est plus un
`HoveringStateEnv` lotf : c'est **Genesis** (physique rigide, URDF cf2x) avec masse posée à **40 g**
(`mass_cycle_kg[1]`). Le finetune est celui de la **vraie pipeline `test_cf`** (`finetune_lotf_jax.py`
lancé en sous-processus dans `.venv`), donc on teste la chaîne *de bout en bout*.

## Deux différences avec le notebook lotf — voulues
1. **Plante = Genesis** (pas `HoveringStateEnv` du quad lourd). C'est le vrai banc d'essai sim-to-real.
2. **Résidu = APPRIS** (MLP fit sur le log de vol Genesis via `build_residual_dataset` +
   `fit_residual_ensemble`), pas analytique. Le notebook lotf isolait la *brique BPTT* sur une
   vérité-terrain ; ici on teste **résidu appris + BPTT + hot-swap** ensemble.

## Correspondance des fonctions (vérifiée)
La finetune Genesis appelle exactement les mêmes briques lotf que le notebook BPTT :

| brique | notebook lotf | pipeline Genesis |
|---|---|---|
| env BPTT | `HoveringStateEnv(crazyflie_quad, use_forward_residual=True)` + `MinMax`/`Log`/`VecEnv` | **idem** (`finetune_lotf_jax.build_bptt_context`) |
| politique | `make_lotf_mlp` + `torch_sd_to_flax` | **idem** (`lotf_jax_bridge`) |
| optim BPTT | `bptt.train` + cosine + `clip_by_global_norm` (`CFG['bptt']`) | **idem** (`finetune_lotf_jax.run_bptt`) |
| résidu | `exact_mass_residual` (analytique) | `fit_residual_ensemble` / `create_vec_funcs` (appris) |
| hyperparams | `configs/lotf_config.yaml` + `cf_params.py` | **idem** |

→ Le notebook lotf est littéralement l'inline de `build_bptt_context` + `run_bptt`, au résidu près.

## Visualisation
On **enregistre une vidéo mp4** (caméra Genesis) de chaque vol, affichée en ligne plus bas. C'est le
mode compatible notebook/headless. *(Le viewer 3D live `show_viewer=True` n'existe qu'en exécution
interactive avec écran — utilise plutôt `core/lotf_genesis_eval.py` sans `--no-viewer` pour ça.)*

## Environnements (à lancer dans `genesis_venv`)
Ce notebook tourne dans **`genesis_venv`** (torch + genesis). Le finetune, lui, tourne dans **`.venv`**
(jax+lotf) appelé en **sous-processus** — exactement comme `lotf_genesis_eval.py --online-finetune`.
```
../../genesis_venv/bin/jupyter nbconvert --to notebook --execute --inplace \
    --ExecutePreprocessor.kernel_name=genesis_venv exact_residual_finetune_genesis.ipynb
```

## 0. Imports et chemins

In [ ]:
import sys, os, subprocess, time
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import Video, display
import genesis as gs

THIS = Path.cwd()
REPO = next(p for p in [THIS, *THIS.parents] if (p / "lotf").is_dir())
TEST_CF = REPO / "test_cf"
CORE = TEST_CF / "core"
REAL = TEST_CF / "RL-real"
for p in (CORE, REAL):
    sys.path.insert(0, str(p))

import cf_params as P
from lotf_config import CFG

%matplotlib inline

## 1. Configuration
Mêmes constantes que le notebook lotf (sources `cf_params.py` / `configs/lotf_config.yaml`).
Le finetune est délégué au Python de `.venv` (jax+lotf) en sous-processus.

In [ ]:
M_NOM  = P.MASS                                       # 0.027 kg
M_REAL = CFG["genesis_bridge"]["mass_cycle_kg"][1]    # 0.040 kg — masse 'lourde' Genesis
GOAL   = list(P.HOVER_GOAL)                           # [0, 0, 0.5]
STEPS  = 600                                          # 12 s @ 50 Hz
FT_EPOCHS = 200                                       # tir unique (validation) ; en ligne 30/step
JAX_PY = str(REPO / ".venv" / "bin" / "python")
VIDEO_DIR = TEST_CF / "data"; VIDEO_DIR.mkdir(exist_ok=True)

print(f"masse Genesis  = {M_REAL*1000:.0f} g  ->  hover {9.81*M_REAL:.3f} N")
print(f"politique 'croit' hover = {9.81*M_NOM:.3f} N (modèle 27 g) -> sous-poussée attendue")
print(f"finetune via : {JAX_PY}")

## 2. Genesis + environnement (une seule fois)
`gs.init` ne peut être appelé qu'une fois par kernel. On construit la scène avec **caméra**
(`visualize_camera=True`) pour pouvoir enregistrer les vidéos, et la cible visible.

In [ ]:
gs.init(logging_level="warning")
from lotf_genesis_env import LotfHoverEnv          # import après gs.init
from RL_policy import RLPolicy
from drone_state import _quat_wxyz_to_R

env = LotfHoverEnv(goal=GOAL, show_viewer=False, visualize_target=True, visualize_camera=True)
print(f"env prêt — masse URDF par défaut = {env.mass*1000:.0f} g  | caméra = {env.cam is not None}")

## 3. Helper de vol (avec enregistrement vidéo optionnel)
Reproduit la boucle de `lotf_genesis_eval.main` : pose la masse, reset, déroule la politique, logge le
rollout au format `finetune_lotf` (`t,p,R,v,T_N,omega`). Si `record_to` est fourni, enregistre un mp4.

In [ ]:
def fly(ckpt_path, mass, steps=STEPS, seed=0, record_to=None):
    np.random.seed(seed)                  # reset() tire une pose initiale -> reproductible
    env.set_total_mass(mass)
    with torch.device("cpu"):
        agent = RLPolicy(str(ckpt_path))
    obs, _ = env.reset()
    log = {"t": [], "p": [], "R": [], "v": [], "T_N": [], "omega": []}
    errs = []
    recording = record_to is not None and env.cam is not None
    if recording:
        env.cam.start_recording()
    with torch.no_grad():
        for i in range(steps):
            a = agent.get_action(obs)
            obs, info = env.step(a)
            if recording:
                env.cam.render()
            errs.append(info["pos_error"])
            an = a.detach().cpu().numpy() if torch.is_tensor(a) else np.asarray(a)
            lat = env.state.latest
            log["t"].append(i * P.DT)
            log["p"].append(lat["pos"].copy())
            log["R"].append(_quat_wxyz_to_R(lat["quat_wxyz"]).flatten())
            log["v"].append(lat["vel"].copy())
            log["T_N"].append(float(an[0]))
            log["omega"].append(an[1:].copy())
    if recording:
        env.cam.stop_recording(save_to_filename=str(record_to), fps=int(1 / P.DT))
        print(f"vidéo -> {record_to}")
    out = {k: np.asarray(v) for k, v in log.items()}
    return out, np.asarray(errs)

## 4. Baseline — politique nominale dans Genesis à 40 g → offset en z (+ vidéo)

In [ ]:
vid_base = VIDEO_DIR / "genesis_base_40g.mp4"
log_base, errs_base = fly(TEST_CF / "models" / "model_pretrain.pt", M_REAL, record_to=vid_base)
tb, zb = log_base["t"], log_base["p"][:, 2]
z_base = zb[-100:].mean()
print(f"baseline  z_final \u2248 {z_base:.3f} m   (cible {GOAL[2]}, offset {z_base-GOAL[2]:+.3f} m)")
print(f"          err pos moyenne (2 dern. s) = {errs_base[-100:].mean():.3f} m")

# log sauvegardé pour le finetune (format finetune_lotf)
log_path = TEST_CF / "measurements" / "online_ft" / "genesis_demo_log.npz"
log_path.parent.mkdir(exist_ok=True)
np.savez_compressed(log_path, **log_base)
print(f"log -> {log_path}")

## 5. Finetune — vraie pipeline `test_cf` en sous-processus (`.venv`)
On appelle `core/finetune_lotf_jax.py` (résidu APPRIS sur le log Genesis + BPTT lotf, mêmes fonctions
que le notebook lotf) avec le Python de `.venv`. Sortie : un `.pt` torch revolable dans Genesis.

In [ ]:
out_pt = TEST_CF / "models" / "model_ft_genesis_demo.pt"
cmd = [JAX_PY, str(CORE / "finetune_lotf_jax.py"),
       "--base", str(TEST_CF / "models" / "model_pretrain.pt"),
       "--log", str(log_path), "--out", str(out_pt),
       "--target", *map(str, GOAL), "--epochs", str(FT_EPOCHS),
       "--window-sec", str(CFG["online"]["window_sec"])]
print(" ".join(cmd), "\n")
t0 = time.time()
res = subprocess.run(cmd, capture_output=True, text=True)
print(res.stdout[-1500:])
if res.returncode != 0:
    print("--- STDERR ---\n", res.stderr[-1500:])
print(f"\n[finetune] code={res.returncode}  en {time.time()-t0:.1f}s")

## 6. Éval — politique finetunée dans Genesis à 40 g → offset corrigé (+ vidéo)

In [ ]:
vid_ft = VIDEO_DIR / "genesis_ft_40g.mp4"
log_ft, errs_ft = fly(out_pt, M_REAL, record_to=vid_ft)
tf, zf = log_ft["t"], log_ft["p"][:, 2]
z_ft = zf[-100:].mean()
print(f"finetunée z_final \u2248 {z_ft:.3f} m   (cible {GOAL[2]}, offset {z_ft-GOAL[2]:+.3f} m)")
print(f"          err pos moyenne (2 dern. s) = {errs_ft[-100:].mean():.3f} m")

## 7. Vidéos des deux vols
Gauche/haut = baseline (le drone reste ~10 cm sous la cible) ; bas = finetunée (revient à la cible).

In [ ]:
print("BASELINE @ 40 g (politique nominale, non adaptée)")
display(Video(str(vid_base), embed=True, width=480))
print("FINETUNÉE @ 40 g (résidu appris + BPTT)")
display(Video(str(vid_ft), embed=True, width=480))

## 8. Comparaison de l'altitude

In [ ]:
plt.figure(figsize=(8, 5))
plt.axhline(GOAL[2], color="k", ls="--", lw=1, label=f"cible z = {GOAL[2]}")
plt.plot(tb, zb, color="#e74c3c", lw=2, label=f"base @ 40 g (z\u2248{z_base:.3f})")
plt.plot(tf, zf, color="#27ae60", lw=2, label=f"finetunée @ 40 g (z\u2248{z_ft:.3f})")
plt.xlabel("temps (s)"); plt.ylabel("altitude z (m)")
plt.title("Genesis 40 g : offset z corrigé par finetune (résidu appris, même BPTT que le notebook lotf)")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print(f"VERDICT : offset {z_base-GOAL[2]:+.3f} m  ->  {z_ft-GOAL[2]:+.3f} m")

## 9. Conclusion

Si l'offset z se réduit nettement, la chaîne complète **(log Genesis → résidu appris → BPTT lotf →
hot-swap → revol Genesis)** est validée *de bout en bout*, avec **exactement les mêmes fonctions** que
le notebook lotf (cf. tableau de correspondance en tête).

Comparaison utile entre les deux notebooks :
- **`exact_residual_finetune_cf.ipynb`** : BPTT seul, résidu *exact* → isole la brique d'optimisation.
- **ce notebook** : résidu *appris* + Genesis → ajoute l'apprentissage du résidu et le mismatch
  sim-to-sim. Un écart entre les deux pointe vers l'apprentissage du résidu ou le mismatch, pas le BPTT.